In [ ]:
# P2PNet for Cell Detection - Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils import data
import torchvision
from torchvision.models import vgg16_bn, resnet50
import numpy as np
import cv2
import os
import yaml
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from scipy.optimize import linear_sum_assignment
import glob

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 파라미터 로드
with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

# 7 클래스 cell type 정의
class_names = {
    0: "Epithelial",
    1: "Stromal", 
    2: "Lymphocyte",
    3: "Plasma",
    4: "Neutrophil",
    5: "Eosinophil",
}

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(class_names.values())}")


In [ ]:
# P2PNet Model Architecture (SOTA)
class P2PNet(nn.Module):
    """
    Point-to-Point Network for Cell Detection
    - Backbone: VGG16-BN or ResNet50
    - Head: Point regression + classification
    - Output: Direct point predictions (no anchors, no heatmaps)
    """
    def __init__(self, num_classes=6, backbone='vgg16_bn', row=2, line=2):
        super(P2PNet, self).__init__()
        self.num_classes = num_classes
        self.row = row
        self.line = line
        
        # Backbone
        if backbone == 'vgg16_bn':
            vgg = vgg16_bn(pretrained=True)
            self.features = nn.Sequential(*list(vgg.features.children())[:-1])
            in_channels = 512
        elif backbone == 'resnet50':
            resnet = resnet50(pretrained=True)
            self.features = nn.Sequential(
                resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
            )
            in_channels = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
        # Regression head (point coordinates)
        self.reg_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * 2, 1)  # (x, y) per grid cell
        )
        
        # Classification head
        self.cls_head = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, row * line * num_classes, 1)  # class scores per grid cell
        )
        
    def forward(self, x):
        """
        Args:
            x: (B, 3, H, W) input images
        Returns:
            points: (B, N, 2) predicted point coordinates (normalized [0, 1])
            logits: (B, N, num_classes) classification logits
        """
        batch_size = x.size(0)
        
        # Backbone
        features = self.features(x)  # (B, C, H_feat, W_feat)
        
        # Regression: predict point offsets
        pred_points = self.reg_head(features)  # (B, row*line*2, H_feat, W_feat)
        pred_points = pred_points.permute(0, 2, 3, 1)  # (B, H_feat, W_feat, row*line*2)
        
        # Classification: predict class scores
        pred_logits = self.cls_head(features)  # (B, row*line*num_classes, H_feat, W_feat)
        pred_logits = pred_logits.permute(0, 2, 3, 1)  # (B, H_feat, W_feat, row*line*num_classes)
        
        h_feat, w_feat = pred_points.shape[1], pred_points.shape[2]
        
        # Reshape to (B, N, 2) and (B, N, num_classes)
        pred_points = pred_points.reshape(batch_size, h_feat * w_feat * self.row * self.line, 2)
        pred_logits = pred_logits.reshape(batch_size, h_feat * w_feat * self.row * self.line, self.num_classes)
        
        # Apply sigmoid to get normalized coordinates [0, 1]
        pred_points = torch.sigmoid(pred_points)
        
        return pred_points, pred_logits


# 모델 초기화
model = P2PNet(num_classes=num_classes, backbone='vgg16_bn', row=2, line=2).to(device)
print(f"\n✅ P2PNet model created")
print(f"  Backbone: VGG16-BN")
print(f"  Number of classes: {num_classes}")
print(f"  Grid: 2x2 per feature map cell")

# 모델 파라미터 수 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")


In [ ]:
# P2PNet Loss Function with Hungarian Matching
class P2PNetLoss(nn.Module):
    """
    P2PNet Loss with Hungarian Algorithm matching
    - Point L1 loss (regression)
    - Focal loss (classification)
    - Hungarian matching for optimal assignment
    """
    def __init__(self, num_classes=7, point_loss_coef=1.0, class_loss_coef=1.0):
        super(P2PNetLoss, self).__init__()
        self.num_classes = num_classes
        self.point_loss_coef = point_loss_coef
        self.class_loss_coef = class_loss_coef
        
    def focal_loss(self, logits, targets, alpha=0.25, gamma=2.0):
        """
        Focal Loss for classification
        Args:
            logits: (N, num_classes) predicted logits
            targets: (N,) target class indices
            alpha: weighting factor
            gamma: focusing parameter
        Returns:
            loss: scalar
        """
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = alpha * (1 - pt) ** gamma * ce_loss
        return focal_loss.mean()
    
    def hungarian_matching(self, pred_points, pred_logits, gt_points, gt_classes):
        """
        Hungarian algorithm for optimal point matching
        Args:
            pred_points: (N_pred, 2) predicted points [0, 1]
            pred_logits: (N_pred, num_classes) predicted class logits
            gt_points: (N_gt, 2) ground truth points [0, 1]
            gt_classes: (N_gt,) ground truth class indices
        Returns:
            matched_pred_idx: (N_gt,) indices of matched predictions
            matched_gt_idx: (N_gt,) indices of ground truth (0, 1, 2, ...)
        """
        if len(gt_points) == 0:
            return torch.tensor([]).long(), torch.tensor([]).long()
        
        # Compute cost matrix: point L1 distance + classification cost
        # Point cost: L1 distance
        point_cost = torch.cdist(pred_points, gt_points, p=1)  # (N_pred, N_gt)
        
        # Classification cost: negative log probability
        pred_probs = F.softmax(pred_logits, dim=-1)  # (N_pred, num_classes)
        class_cost = -pred_probs[:, gt_classes]  # (N_pred, N_gt)
        
        # Total cost
        cost_matrix = point_cost + class_cost  # (N_pred, N_gt)
        
        # Hungarian matching
        cost_matrix_np = cost_matrix.detach().cpu().numpy()
        pred_idx, gt_idx = linear_sum_assignment(cost_matrix_np)
        
        return torch.from_numpy(pred_idx).long(), torch.from_numpy(gt_idx).long()
    
    def forward(self, pred_points, pred_logits, targets):
        """
        Args:
            pred_points: (B, N, 2) predicted points [0, 1]
            pred_logits: (B, N, num_classes) predicted class logits
            targets: dict with 'points' (B, M, 2) and 'classes' (B, M)
        Returns:
            total_loss: scalar
            loss_dict: dict with individual losses
        """
        batch_size = pred_points.size(0)
        device = pred_points.device
        
        total_point_loss = 0
        total_class_loss = 0
        num_matched = 0
        
        for b in range(batch_size):
            gt_points = targets['points'][b]  # (M, 2)
            gt_classes = targets['classes'][b].long()  # (M,)
            
            # Remove invalid targets (padding)
            valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
            gt_points = gt_points[valid_mask]
            gt_classes = gt_classes[valid_mask]
            
            if len(gt_points) == 0:
                continue
            
            # Hungarian matching
            pred_idx, gt_idx = self.hungarian_matching(
                pred_points[b], pred_logits[b], gt_points, gt_classes
            )
            
            if len(pred_idx) == 0:
                continue
            
            # Matched predictions and targets
            matched_pred_points = pred_points[b][pred_idx]
            matched_pred_logits = pred_logits[b][pred_idx]
            matched_gt_points = gt_points[gt_idx]
            matched_gt_classes = gt_classes[gt_idx]
            
            # Point L1 loss
            point_loss = F.l1_loss(matched_pred_points, matched_gt_points, reduction='sum')
            
            # Focal loss for classification
            class_loss = self.focal_loss(matched_pred_logits, matched_gt_classes)
            
            total_point_loss += point_loss
            total_class_loss += class_loss * len(pred_idx)
            num_matched += len(pred_idx)
        
        # Average losses
        if num_matched > 0:
            total_point_loss = total_point_loss / num_matched
            total_class_loss = total_class_loss / num_matched
        else:
            total_point_loss = torch.tensor(0.0, device=device)
            total_class_loss = torch.tensor(0.0, device=device)
        
        # Total loss
        total_loss = self.point_loss_coef * total_point_loss + self.class_loss_coef * total_class_loss
        
        loss_dict = {
            'total': total_loss.item(),
            'point': total_point_loss.item(),
            'class': total_class_loss.item(),
            'num_matched': num_matched
        }
        
        return total_loss, loss_dict


# Loss function 초기화
criterion = P2PNetLoss(num_classes=num_classes, point_loss_coef=1.0, class_loss_coef=1.0).to(device)
print(f"\n✅ P2PNet loss function created")
print(f"  Point loss coefficient: 1.0")
print(f"  Class loss coefficient: 1.0")
print(f"  Matching: Hungarian algorithm")


In [ ]:
# Data Loading (from yolov11_train.ipynb)
input_size = 512
label_dir = '../../data/spatialTranscriptome/detail_preprocessed_xenium/labels/'
label_files = sorted(glob.glob(f"{label_dir}/*.csv"))

image_filenames = []
labels = []

# 라벨 파일 로드
print("📂 Loading labels...")
for label_file in tqdm(label_files):
    data_df = pd.read_csv(label_file)
    
    # CSV 파일명에서 이미지 경로 생성
    img_name = os.path.basename(label_file).replace('.csv', '.png')
    img_path = label_file.replace('/labels/', '/patches/').replace('.csv', '.png')
    
    if os.path.exists(img_path):
        image_filenames.append(img_path)
        
        # 중심점 계산 및 정규화
        centers = []
        classes = []
        for _, row in data_df.iterrows():
            cx = (row['x1'] + row['x2']) / 2
            cy = (row['y1'] + row['y2']) / 2
            
            # 클래스 매핑
            class_name = row['class_name']
            class_id = [k for k, v in class_names.items() if v == class_name]
            if class_id:
                centers.append([cx, cy])
                classes.append(class_id[0])
        
        if len(centers) > 0:
            labels.append({
                'points': np.array(centers, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int64)
            })
        else:
            image_filenames.pop()

print(f"✅ Loaded {len(image_filenames)} images with labels")

# 이미지 로드
print("\n📷 Loading images...")
images = []
for img_path in tqdm(image_filenames):
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 512x512로 패딩 (필요한 경우)
    if image.shape[0] != 512 or image.shape[1] != 512:
        image = cv2.copyMakeBorder(
            image, 0, 512 - image.shape[0], 0, 512 - image.shape[1],
            cv2.BORDER_CONSTANT, value=[255, 255, 255]
        )
    
    images.append(image)

print(f"✅ Loaded {len(images)} images")
print(f"  Image shape: {images[0].shape}")
print(f"  Label example - Points: {labels[0]['points'].shape}, Classes: {labels[0]['classes'].shape}")


In [ ]:
# Custom Dataset for P2PNet
class P2PNetDataset(data.Dataset):
    def __init__(self, images, labels, img_size=512, augment=False, max_points=500):
        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.augment = augment
        self.max_points = max_points  # 최대 포인트 수 (패딩용)
        self.n = len(self.images)
    
    def __len__(self):
        return self.n
    
    def __getitem__(self, index):
        # 이미지 로드
        image = self.images[index].copy()
        points = self.labels[index]['points'].copy()  # (N, 2) in pixel coordinates
        classes = self.labels[index]['classes'].copy()  # (N,)
        
        h, w = image.shape[:2]
        
        # Augmentation
        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                image = np.fliplr(image).copy()
                points[:, 0] = w - points[:, 0]
            
            # Vertical flip
            if random.random() < 0.5:
                image = np.flipud(image).copy()
                points[:, 1] = h - points[:, 1]
        
        # Normalize points to [0, 1]
        points[:, 0] = points[:, 0] / w
        points[:, 1] = points[:, 1] / h
        
        # Normalize image
        image = image.astype(np.float32) / 255.0
        
        # HWC -> CHW
        image = image.transpose((2, 0, 1))
        
        # Padding to max_points
        num_points = len(points)
        if num_points < self.max_points:
            padded_points = np.full((self.max_points, 2), -1.0, dtype=np.float32)
            padded_classes = np.full((self.max_points,), -1, dtype=np.int64)
            
            padded_points[:num_points] = points
            padded_classes[:num_points] = classes
            
            points = padded_points
            classes = padded_classes
        else:
            # Truncate if too many points
            points = points[:self.max_points]
            classes = classes[:self.max_points]
        
        return (torch.from_numpy(image).float(),
                torch.from_numpy(points).float(),
                torch.from_numpy(classes).long())


def collate_fn_p2pnet(batch):
    """Collate function for P2PNet"""
    images, points, classes = zip(*batch)
    
    images = torch.stack(images, dim=0)  # (B, 3, H, W)
    points = torch.stack(points, dim=0)  # (B, max_points, 2)
    classes = torch.stack(classes, dim=0)  # (B, max_points)
    
    targets = {
        'points': points,
        'classes': classes
    }
    
    return images, targets


# Train/Val split
train_images, val_images, train_labels, val_labels = train_test_split(
    images, labels, test_size=0.1, random_state=242, shuffle=True
)

# Create datasets
train_dataset = P2PNetDataset(train_images, train_labels, augment=True, max_points=500)
val_dataset = P2PNetDataset(val_images, val_labels, augment=False, max_points=500)

print(f"\n📊 Dataset split:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Max points per image: 500")

# Create dataloaders
batch_size = 8
train_loader = data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)
val_loader = data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn_p2pnet, pin_memory=True
)

print(f"\n📦 Dataloaders created:")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")


In [ ]:
# Visualization function
def visualize_p2pnet_sample(dataset, index=0):
    """Visualize a sample from P2PNet dataset"""
    image_tensor, points_tensor, classes_tensor = dataset[index]
    
    # Convert to numpy
    image = image_tensor.numpy().transpose(1, 2, 0)  # CHW -> HWC
    points = points_tensor.numpy()
    classes = classes_tensor.numpy()
    
    # Filter valid points
    valid_mask = (points[:, 0] >= 0) & (points[:, 1] >= 0)
    points = points[valid_mask]
    classes = classes[valid_mask]
    
    # Denormalize points
    h, w = image.shape[:2]
    points_pixel = points.copy()
    points_pixel[:, 0] *= w
    points_pixel[:, 1] *= h
    
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image)
    
    # Color map
    colors = ['red', 'green', 'yellow', 'magenta', 'dodgerblue', 'orange', 'gray']
    
    for i in range(len(points_pixel)):
        x, y = points_pixel[i]
        class_id = int(classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        
        # Draw circle
        circle = plt.Circle((x, y), 3, color=color, fill=True, alpha=0.7)
        ax.add_patch(circle)
    
    ax.set_title(f'Sample {index} - Total points: {len(points)}', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Legend
    legend_elements = [
        patches.Patch(color=colors[i], label=f'{class_names[i]}: {sum(classes==i)}')
        for i in range(num_classes) if sum(classes==i) > 0
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Sample {index} statistics:")
    print(f"  Total points: {len(points)}")
    for i in range(num_classes):
        count = sum(classes == i)
        if count > 0:
            print(f"  {class_names[i]}: {count}")

# Visualize a sample
print("🖼️ Visualizing training sample...")
visualize_p2pnet_sample(train_dataset, index=5)


In [ ]:
# Training Setup
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Scheduler
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# Training parameters
epochs = 1000
save_dir = '../../model/spatialTranscriptome/p2pnet/'
os.makedirs(save_dir, exist_ok=True)

# Tracking metrics
train_losses = []
val_point_errors = []
val_class_accs = []
best_val_acc = 0

print(f"\n🚀 Training setup:")
print(f"  Epochs: {epochs}")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: CosineAnnealingWarmRestarts")
print(f"  Save directory: {save_dir}")


In [ ]:
# Training Loop
print("\n" + "="*80)
print("🚀 Starting P2PNet Training")
print("="*80)

for epoch in range(epochs):
    # ==================== TRAINING ====================
    model.train()
    train_loss = 0
    train_point_loss = 0
    train_class_loss = 0
    train_matched = 0
    
    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                     desc=f'Epoch {epoch+1}/{epochs} [Train]')
    
    for batch_idx, (images, targets) in train_pbar:
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        
        # Forward
        pred_points, pred_logits = model(images)
        
        # Loss
        loss, loss_dict = criterion(pred_points, pred_logits, targets)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Update metrics
        train_loss += loss_dict['total']
        train_point_loss += loss_dict['point']
        train_class_loss += loss_dict['class']
        train_matched += loss_dict['num_matched']
        
        # Update progress bar
        memory = f'{torch.cuda.memory_reserved() / 1E9:.2f}G'
        train_pbar.set_postfix({
            'loss': f"{loss_dict['total']:.4f}",
            'pt': f"{loss_dict['point']:.4f}",
            'cls': f"{loss_dict['class']:.4f}",
            'mem': memory
        })
    
    # Average training metrics
    num_batches = len(train_loader)
    avg_train_loss = train_loss / num_batches
    avg_train_point_loss = train_point_loss / num_batches
    avg_train_class_loss = train_class_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # ==================== VALIDATION ====================
    model.eval()
    val_point_error = 0
    val_class_correct = 0
    val_total = 0
    
    val_pbar = tqdm(enumerate(val_loader), total=len(val_loader),
                   desc=f'Epoch {epoch+1}/{epochs} [Val]')
    
    with torch.no_grad():
        for batch_idx, (images, targets) in val_pbar:
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}
            
            # Forward
            pred_points, pred_logits = model(images)
            
            # Compute metrics per batch
            batch_size = images.size(0)
            for b in range(batch_size):
                gt_points = targets['points'][b]
                gt_classes = targets['classes'][b].long()
                
                # Filter valid points
                valid_mask = (gt_points[:, 0] >= 0) & (gt_points[:, 1] >= 0)
                gt_points = gt_points[valid_mask]
                gt_classes = gt_classes[valid_mask]
                
                if len(gt_points) == 0:
                    continue
                
                # Hungarian matching
                from scipy.optimize import linear_sum_assignment
                point_cost = torch.cdist(pred_points[b], gt_points, p=1)
                pred_probs = F.softmax(pred_logits[b], dim=-1)
                class_cost = -pred_probs[:, gt_classes]
                cost_matrix = (point_cost + class_cost).detach().cpu().numpy()
                
                pred_idx, gt_idx = linear_sum_assignment(cost_matrix)
                
                # Point error
                matched_pred_points = pred_points[b][pred_idx]
                matched_gt_points = gt_points[gt_idx]
                point_error = torch.abs(matched_pred_points - matched_gt_points).mean()
                val_point_error += point_error.item()
                
                # Classification accuracy
                matched_pred_classes = pred_logits[b][pred_idx].argmax(dim=-1)
                matched_gt_classes = gt_classes[gt_idx]
                correct = (matched_pred_classes == matched_gt_classes).sum().item()
                val_class_correct += correct
                val_total += len(gt_idx)
    
    # Average validation metrics
    avg_val_point_error = val_point_error / len(val_loader)
    avg_val_class_acc = val_class_correct / val_total if val_total > 0 else 0
    val_point_errors.append(avg_val_point_error)
    val_class_accs.append(avg_val_class_acc)
    
    # Scheduler step
    scheduler.step()
    
    # ==================== LOGGING ====================
    print(f"\n{'='*80}")
    print(f"Epoch {epoch+1}/{epochs} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f} (Point: {avg_train_point_loss:.4f}, Class: {avg_train_class_loss:.4f})")
    print(f"  Val Point Error: {avg_val_point_error:.4f}")
    print(f"  Val Class Accuracy: {avg_val_class_acc:.4f}")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*80}\n")
    
    # ==================== SAVE CHECKPOINT ====================
    # Save best model
    if avg_val_class_acc > best_val_acc:
        best_val_acc = avg_val_class_acc
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': avg_train_loss,
            'val_point_error': avg_val_point_error,
            'val_class_acc': avg_val_class_acc,
            'best_val_acc': best_val_acc
        }
        torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 New best model saved! Val Acc: {avg_val_class_acc:.4f}\n")
    
    # Save last model
    last_checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': avg_train_loss,
        'val_point_error': avg_val_point_error,
        'val_class_acc': avg_val_class_acc
    }
    torch.save(last_checkpoint, os.path.join(save_dir, 'last_model.pt'))
    
    # ==================== PLOT PROGRESS ====================
    if (epoch + 1) % 10 == 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Train loss
        axes[0].plot(train_losses, label='Train Loss', color='blue')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Loss')
        axes[0].legend()
        axes[0].grid(True)
        
        # Val point error
        axes[1].plot(val_point_errors, label='Val Point Error', color='red')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Point Error')
        axes[1].set_title('Validation Point Error')
        axes[1].legend()
        axes[1].grid(True)
        
        # Val class accuracy
        axes[2].plot(val_class_accs, label='Val Class Acc', color='green')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Accuracy')
        axes[2].set_title('Validation Classification Accuracy')
        axes[2].legend()
        axes[2].grid(True)
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f'training_progress_epoch_{epoch+1}.png'), dpi=150)
        plt.show()

print("\n" + "="*80)
print("🎯 Training Complete!")
print(f"  Best Val Acc: {best_val_acc:.4f}")
print(f"  Models saved to: {save_dir}")
print("="*80)


In [ ]:
# Evaluation and Visualization
def visualize_predictions(model, dataset, index=0, conf_threshold=0.5, distance_threshold=16):
    """Visualize ground truth and predictions side by side"""
    model.eval()
    
    # Get sample
    image_tensor, gt_points_tensor, gt_classes_tensor = dataset[index]
    
    # Filter valid GT points
    valid_mask = (gt_points_tensor[:, 0] >= 0) & (gt_points_tensor[:, 1] >= 0)
    gt_points = gt_points_tensor[valid_mask].numpy()
    gt_classes = gt_classes_tensor[valid_mask].numpy()
    
    # Predict
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        pred_points, pred_logits = model(image_batch)
        pred_points = pred_points[0].cpu().numpy()  # (N, 2)
        pred_classes = pred_logits[0].argmax(dim=-1).cpu().numpy()  # (N,)
        pred_scores = F.softmax(pred_logits[0], dim=-1).max(dim=-1)[0].cpu().numpy()  # (N,)
    
    # Filter by confidence
    conf_mask = pred_scores > conf_threshold
    pred_points = pred_points[conf_mask]
    pred_classes = pred_classes[conf_mask]
    pred_scores = pred_scores[conf_mask]
    
    # Convert image
    image = image_tensor.numpy().transpose(1, 2, 0)  # CHW -> HWC
    h, w = image.shape[:2]
    
    # Denormalize points
    gt_points_pixel = gt_points.copy()
    gt_points_pixel[:, 0] *= w
    gt_points_pixel[:, 1] *= h
    
    pred_points_pixel = pred_points.copy()
    pred_points_pixel[:, 0] *= w
    pred_points_pixel[:, 1] *= h
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    colors = ['red', 'green', 'yellow', 'magenta', 'dodgerblue', 'orange', 'gray']
    
    # Ground Truth
    axes[0].imshow(image)
    for i in range(len(gt_points_pixel)):
        x, y = gt_points_pixel[i]
        class_id = int(gt_classes[i])
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[0].add_patch(circle)
    axes[0].set_title(f'Ground Truth ({len(gt_points_pixel)} points)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Predictions
    axes[1].imshow(image)
    for i in range(len(pred_points_pixel)):
        x, y = pred_points_pixel[i]
        class_id = int(pred_classes[i])
        score = pred_scores[i]
        color = colors[class_id] if class_id < len(colors) else 'white'
        circle = plt.Circle((x, y), 4, color=color, fill=True, alpha=0.8)
        axes[1].add_patch(circle)
        
        # Draw confidence
        if score > 0.7:
            axes[1].text(x+5, y-5, f'{score:.2f}', fontsize=8, color='white',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.7))
    
    axes[1].set_title(f'Predictions ({len(pred_points_pixel)} points, conf>{conf_threshold})',
                     fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Legend
    legend_elements = [
        patches.Patch(color=colors[i], label=class_names[i])
        for i in range(num_classes)
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=num_classes,
              bbox_to_anchor=(0.5, -0.05), fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1)
    plt.show()
    
    # Statistics
    print(f"\n📊 Sample {index} statistics:")
    print(f"  Ground Truth: {len(gt_points_pixel)} points")
    print(f"  Predictions: {len(pred_points_pixel)} points (conf > {conf_threshold})")
    print(f"\n  GT Class distribution:")
    for i in range(num_classes):
        count = sum(gt_classes == i)
        if count > 0:
            print(f"    {class_names[i]}: {count}")
    
    print(f"\n  Pred Class distribution:")
    for i in range(num_classes):
        count = sum(pred_classes == i)
        if count > 0:
            print(f"    {class_names[i]}: {count}")

# Load best model
checkpoint_path = os.path.join(save_dir, 'best_model.pt')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Best model loaded (Epoch {checkpoint['epoch']+1})")
    print(f"  Val Class Acc: {checkpoint.get('val_class_acc', 'N/A')}")

# Visualize predictions
print("\n🖼️ Visualizing predictions...")
sample_idx = random.randint(0, len(val_dataset)-1)
visualize_predictions(model, val_dataset, index=sample_idx, conf_threshold=0.3)
